# Final Project
## Machine Learning for Neuroscience
### Gaia Negev and Tzlil Tabib

In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import ast
from collections import Counter
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from mpl_toolkits.mplot3d import Axes3D

# Initial data handling
loading, organizing and dividing to train and test

In [ ]:
# # load data from json
# with open('data/emoset_challenge_1000_augmented.json', 'r', encoding='utf-8') as f:
#     data = json.load(f)

# # data to pd.DataFrame
# df = pd.DataFrame(data)
# df_annotations = pd.DataFrame(df['annotations'].tolist())
# df = pd.concat([df.drop('annotations', axis=1), df_annotations], axis=1)
# df.head()

# # remove image_name column because it's duplicated in image_id
# df = df.drop('image_name', axis=1)
# # make image_id the index
# df = df.set_index('image_id')
# df.head()

# # divide data to train and test
# train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
# print(f'Train size: {len(train_df)}, Test size: {len(test_df)}')

# # save division to csv
# train_df.to_csv('data/train.csv')
# test_df.to_csv('data/test.csv')

In [ ]:
# load the data 
train_df = pd.read_csv('data/train.csv', index_col='image_id')
test_df = pd.read_csv('data/test.csv', index_col='image_id')

# EDA

In [ ]:
# TODO: K-means, K-modes to explore pattern in the metadata and how it relates to the emotion labels

In [ ]:
train_df.info()

# null analysis
train_df.isnull().sum()

### Plan 
division of variables by type 
numeric - brightness, colorfulness 
category - facial expression, emotion
text - description, viewer_feelings, object, human_action, scene

Process: 
1. numeric - standarize
2. category - binary coding
3. tfidf for text variables 
4. dimension reduction to all metadata
5. insert to a model description, feelings and metadata 
    a. model selection 
    b. regularization
    c. interpretation

In [ ]:
# print how many unique values are in each column
for column in train_df.columns:
    unique_values = train_df[column].nunique()
    print(f'{column}: {unique_values} unique values')

In [ ]:
# emotion distribution to see if there is a class imbalance
emotion_counts = train_df['emotion'].value_counts()
print(emotion_counts)
emotion_counts.plot(kind='bar')

Brightness and colorfulness have a very low percentage of missing values, so we'll consider removing them depending on the model (the model's ability to work with missing values).

In [ ]:
# TODO: decide how to handle missing values in the metadata columns

## Metadata Analysis:

In [ ]:
def analyze_column(df, col_name, target_col='emotion'):
    """
    Performs a distribution analysis, null check, and correlation 
    with a target column for a specific feature.
    """
    print(f"\n{'='*10} Analyzing Column: {col_name} {'='*10}")
    
    # 0. fill 'missing' in null values for better visualization
    df[col_name] = df[col_name].fillna('missing')
    
    # 1. Distribution of the column
    plt.figure(figsize=(8, 4))
    df[col_name].value_counts().plot(kind='bar')
    plt.title(f'{col_name.replace("_", " ").title()} Distribution')
    plt.xlabel(col_name)
    plt.ylabel('Count')
    plt.show()

    # 2. Null analysis
    null_entries = df[df[col_name].isnull()]
    print(f"Number of null {col_name} entries: {len(null_entries)}")

    if len(null_entries) > 0:
        plt.figure(figsize=(6, 6))
        null_entries[target_col].value_counts().plot(kind='pie', autopct='%1.1f%%')
        plt.title(f'{target_col.title()} distribution when {col_name} is Null')
        plt.show()
    else:
        print(f"No null values found in {col_name}.")

    # 3. Correlation / Cross-tabulation with Target
    non_null_df = df[df[col_name].notnull()]
    if not non_null_df.empty:
        correlation = pd.crosstab(non_null_df[col_name], non_null_df[target_col])
        print(f"\nCross-tabulation ({col_name} vs {target_col}):")
        print(correlation)
        
        correlation.plot(kind='bar', stacked=True, figsize=(10, 6))
        plt.title(f'{col_name.title()} vs {target_col.title()}')
        plt.xlabel(col_name)
        plt.ylabel('Count')
        plt.legend(title=target_col, bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.show()

for column in ['facial_expression',  'human_action', 'scene']:
    analyze_column(train_df, column)

## object
maybe it's better to address it as text instead of categorial? adding 130 variables (with mostly 0) to 800 entries might lead to overfitting 

In [ ]:
# 1. Clean and convert to lists
def clean_to_list(val):
    if isinstance(val, list):
        return val
    if not isinstance(val, str) or val == 'missing':
        return []
    
    # If it looks like a Python list string "['a', 'b']", evaluate it
    if val.startswith('[') and val.endswith(']'):
        try:
            return ast.literal_eval(val)
        except:
            return []
    
    # If it's a comma-separated string "Sunglasses,Flower", split it
    return [item.strip() for item in val.split(',')]

# Apply the conversion
train_df['object_list'] = train_df['object'].fillna('missing').apply(clean_to_list)

# 2. Flatten the list and count occurrences
all_objects = []
for lst in train_df['object_list']:
    all_objects.extend(lst)

object_counts = Counter(all_objects)

# Result
print(f"Count for 'Plant': {object_counts['Plant']}")
print(f"Count for 'Flower': {object_counts['Flower']}")

# convert obect_count to dataframe for better visualization
object_counts_df = pd.DataFrame.from_dict(object_counts, orient='index', columns=['count'])
object_counts_df = object_counts_df.sort_values(by='count', ascending=False)
print(object_counts_df)

# Preprocess Variables

## Numerics Variables

In [ ]:
train_df.describe()

# standardize the numeric variables - brightness and colorfulness
scaler = StandardScaler()
train_df[['standardized_brightness', 'standardized_colorfulness']] = scaler.fit_transform(train_df[['brightness', 'colorfulness']]) 

# extract the parameters mean and std to use in the model
train_means = train_df[['standardized_brightness', 'standardized_colorfulness']].mean()
train_stds = train_df[['standardized_brightness', 'standardized_colorfulness']].std()

print(f"Correlation between standardized brightness and colorfulness: {train_df['standardized_brightness'].corr(train_df['standardized_colorfulness']):.2f}")

# check correlation and visualizebetween brightness and colorfulness
plt.figure(figsize=(8, 6))
plt.scatter(train_df['standardized_brightness'], train_df['standardized_colorfulness'], alpha=0.5)
plt.title('Brightness vs Colorfulness')
plt.xlabel('Brightness (standardized)')
plt.ylabel('Colorfulness (standardized)')
plt.show()

# plot raw brightness and colorfulness
plt.figure(figsize=(8, 6))
plt.scatter(train_df['brightness'], train_df['colorfulness'], alpha=0.5)
plt.title('Brightness vs Colorfulness (Raw)')
plt.xlabel('Brightness (raw)')
plt.ylabel('Colorfulness (raw)')
plt.show()

### Conclusions from metadata analysis and EDA: 

low correlation between brightness and colorfulness, keep both 
89% of the samples are missing facial expression
We checked the correlation with emotion to see if we can learn anything about it, we can't. 
Looks like there's no null pattern we can manually overcome
Since most of the sample is missing this field, we'll consider to leave this variable out of the analysis.  

In [ ]:
# TODO: rephrase conclusion about the metadata analysis

## Labeling categorial variables - facial_expression and emotion

In [ ]:
# Target: Label Encode 'emotion' [cite: 6, 28]
le = LabelEncoder()
train_df['target'] = le.fit_transform(train_df['emotion'])
mapping = dict(zip(le.classes_, range(len(le.classes_))))
print(f"Emotion Mapping (Target): {mapping}")

# Features: One-Hot Encode 'facial_expression' 
# We use a fresh encoder here so it doesn't conflict with anything else
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
facial_condined = train_df['facial_expression'].fillna('missing').values.reshape(-1, 1)
facial_encoded = ohe.fit_transform(facial_condined)

# Convert to dataframe with clear column names for your Feature Analysis [cite: 12]
facial_expression_df = pd.DataFrame(
    facial_encoded, 
    columns=ohe.get_feature_names_out(['facial_expression']), 
    index=train_df.index
)


## TFIDF

In [ ]:
# TODO: hyperparameter tuning - how many max features to select per variable?

In [ ]:

# 1. Fill NaN values with empty strings to prevent errors
text_columns = ['description', 'viewer_feelings', 'object', 'scene', 'human_action']
train_df[text_columns] = train_df[text_columns].fillna('')

# 2. TF-IDF for long-form text (Description & Viewer Feelings)
# We limit to 200 features because we have 800 training samples
tfidf_desc = TfidfVectorizer(stop_words='english', max_features=200)
tfidf_feel = TfidfVectorizer(stop_words='english', max_features=200)

desc_matrix = tfidf_desc.fit_transform(train_df['description'])
feel_matrix = tfidf_feel.fit_transform(train_df['viewer_feelings'])

# 3. TF-IDF for short metadata tags (Object, Scene, Action)
# These usually have a smaller vocabulary
tfidf_obj = TfidfVectorizer(max_features=100)
tfidf_scene = TfidfVectorizer(max_features=50)
tfidf_action = TfidfVectorizer(max_features=50, ngram_range=(1, 2)) # to capture bigrams like "playing guitar"

obj_matrix = tfidf_obj.fit_transform(train_df['object'])
scene_matrix = tfidf_scene.fit_transform(train_df['scene'])
action_matrix = tfidf_action.fit_transform(train_df['human_action'])

In [ ]:
# present the first 5 rows as a dataframe with feature names, of the description, feelings, ogbject, scene, human_action tf-idf matrix separatley

desc_df = pd.DataFrame(desc_matrix[:5].toarray(), columns=tfidf_desc.get_feature_names_out())
feel_df = pd.DataFrame(feel_matrix[:5].toarray(), columns=tfidf_feel.get_feature_names_out())
obj_df = pd.DataFrame(obj_matrix[:5].toarray(), columns=tfidf_obj.get_feature_names_out())
scene_df = pd.DataFrame(scene_matrix[:5].toarray(), columns=tfidf_scene.get_feature_names_out())
action_df = pd.DataFrame(action_matrix[:5].toarray(), columns=tfidf_action.get_feature_names_out())


## PCA

In [ ]:
# We will create 5 separate SVD objects to keep the meanings separate
# n_components are chosen based on the complexity of the source
svd_desc = TruncatedSVD(n_components=40)   # Complex descriptions
svd_feel = TruncatedSVD(n_components=40)   # Complex feelings
svd_obj = TruncatedSVD(n_components=15)    # Objects are simpler tags
svd_scene = TruncatedSVD(n_components=5)    # Scenes are very limited
svd_action = TruncatedSVD(n_components=5)   # Actions are short

# Transform the matrices we created in the previous step
desc_pca = svd_desc.fit_transform(desc_matrix)
feel_pca = svd_feel.fit_transform(feel_matrix)
obj_pca = svd_obj.fit_transform(obj_matrix)
scene_pca = svd_scene.fit_transform(scene_matrix)
action_pca = svd_action.fit_transform(action_matrix)

print(f"Total Text/Tag Features: {desc_pca.shape[1] + feel_pca.shape[1] + obj_pca.shape[1] + scene_pca.shape[1] + action_pca.shape[1]}")
# Total should be around 105 features

In [ ]:
# --- PLOT 1: Explained Variance Ratio (Scree Plot) ---
def plot_variance(pca_model):
    # 'pca_model' should be a fitted PCA or TruncatedSVD object
    indv_var = pca_model.explained_variance_ratio_
    cum_var = np.cumsum(indv_var)
    
    plt.figure(figsize=(6, 6))
    plt.plot(range(1, len(indv_var) + 1), indv_var, marker='o', color='purple', label='Individual Component', linewidth=2)
    plt.plot(range(1, len(cum_var) + 1), cum_var, marker='o', color='orange', label='Cumulative', linewidth=2)
    
    plt.title("Explained Variance Ratio by Number of Principal Components")
    plt.xlabel("Number of Principal Components")
    plt.ylabel("Explained Variance Ratio")
    plt.xticks(range(1, len(indv_var) + 1))
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

# --- PLOT 2: 3D Projection ---
def plot_3d_pca(X_pca, labels):
    # X_pca should be a matrix with at least 3 columns
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # 'labels' allows you to color by the actual emotion (amusement, anger, etc.)
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], c=labels, cmap='viridis', alpha=0.6)
    
    ax.set_title("Dataset Projected Over 3 Dimensions Using PCA")
    ax.set_xlabel("Principal Component #1")
    ax.set_ylabel("Principal Component #2")
    ax.set_zlabel("Principal Component #3")
    
    # Adjust view angle to match your example
    ax.view_init(elev=10, azim=-95)
    plt.show()

# --- PLOT 3: 2D Projection ---
def plot_2d_pca(X_pca, labels):
    # X_pca should be a matrix with at least 2 columns
    plt.figure(figsize=(7, 5))
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', alpha=0.9)
    
    plt.title("Dataset Projected Over 2 Dimensions Using PCA")
    plt.xlabel("Principal Component #1")
    plt.ylabel("Principal Component #2")
    plt.show()

# --- HOW TO CALL THEM ---
# Assuming 'y' contains your numeric emotion labels
# and 'svd_desc' is the fitted TruncatedSVD from your previous step
plot_variance(svd_desc)
plot_3d_pca(desc_pca, y)
plot_2d_pca(desc_pca, y)

# modeling - 

inversing to the target label encoding to the model results to allow interpretation

# Embedding

## PCA